# RosenblattBicop demo

This notebook demonstrates conditional pair-copula estimation with `RosenblattBicop`. We simulate observations from a Clayton copula whose Kendall's $\tau$ increases linearly with a scalar covariate $x$, fit the same data with several conditional-density backends, and compare their conditional densities and dependence curves with the known data-generating process.

The pointwise API uses a two-column `uv` matrix throughout. The optional conditioning matrix `x` has one row per `uv` observation.

## Setup

The TabPFN and TabICL backends may require their model credentials or downloads. Install the optional comparison backends with, for example, `uv sync --extra cu130 --extra gbm --extra ngboost --extra tabicl`. Backends that are unavailable or fail to initialize are reported and skipped so the rest of the draft remains runnable.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pyvinecopulib as pv
import torch
from dotenv import load_dotenv
from scipy.stats import norm

from npcc import FitControlsRosenblattBicop, RosenblattBicop

load_dotenv()

SEED = 42
N_TRAIN = 750
TAU_MIN, TAU_MAX = 0.10, 0.80
X_MIN, X_MAX = 0.01, 0.99
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

rng = np.random.default_rng(SEED)
print(f"device: {DEVICE}")

## Conditional Clayton data

For the Clayton family, $\tau = \theta/(\theta+2)$, hence $\theta = 2\tau/(1-\tau)$. We draw independent base uniforms $(u,w)$ and apply the inverse Rosenblatt transform row by row using the Clayton parameter implied by that row's $x$.

In [ ]:
def tau_of_x(x: np.ndarray) -> np.ndarray:
  x = np.asarray(x, dtype=np.float64)
  return TAU_MIN + (TAU_MAX - TAU_MIN) * x


def theta_of_tau(tau: np.ndarray) -> np.ndarray:
  tau = np.asarray(tau, dtype=np.float64)
  return 2.0 * tau / (1.0 - tau)


def clayton_at_tau(tau: float) -> pv.Bicop:
  return pv.Bicop(
    family=pv.BicopFamily.clayton,
    parameters=np.array(
      [[float(theta_of_tau(np.array(tau)))]], dtype=np.float64
    ),
  )


def sample_conditional_clayton(
  n: int, rng: np.random.Generator
) -> tuple[np.ndarray, np.ndarray]:
  x = rng.uniform(X_MIN, X_MAX, size=n)
  u = rng.uniform(1e-9, 1.0 - 1e-9, size=n)
  w = rng.uniform(1e-9, 1.0 - 1e-9, size=n)
  parameters = theta_of_tau(tau_of_x(x))[:, None]
  clayton = pv.Bicop(
    family=pv.BicopFamily.clayton,
    parameters=np.array([[1.0]], dtype=np.float64),
  )
  v = clayton.hinv1(np.column_stack([u, w]), parameters)
  return np.column_stack([u, v]), x[:, None]


uv_train, x_train = sample_conditional_clayton(N_TRAIN, rng)

fig, ax = plt.subplots(figsize=(6, 5))
points = ax.scatter(
  uv_train[:, 0], uv_train[:, 1], c=x_train[:, 0], s=12, alpha=0.65
)
ax.set(xlabel="u", ylabel="v", title="Conditional Clayton training data")
ax.set_aspect("equal")
fig.colorbar(points, ax=ax, label="x")
plt.show()

## Fit several Rosenblatt backends

`tabpfn-criterion` reads TabPFN's native predictive head, while `tabpfn-quantiles`, `gbm`, and `tabicl` expose quantile-based conditional distributions. `ngboost` supplies a parametric probabilistic regressor. All are hosted behind the same `RosenblattBicop` interface.

In [ ]:
BACKENDS = {
  "TabPFN criterion": {
    "backend": "tabpfn-criterion",
  },
  "TabPFN quantiles": {
    "backend": "tabpfn-quantiles",
  },
  "Quantile GBM": {
    "backend": "gbm",
    "backend_kwargs": {
      "n_estimators": 100,
      "max_depth": 3,
      "random_state": SEED,
    },
  },
  "NGBoost": {
    "backend": "ngboost",
    "backend_kwargs": {
      "n_estimators": 300,
      "learning_rate": 0.03,
      "random_state": SEED,
    },
  },
  "TabICL": {
    "backend": "tabicl",
  },
}

models: dict[str, RosenblattBicop] = {}
for label, config in BACKENDS.items():
  try:
    started = perf_counter()
    model = RosenblattBicop(
      FitControlsRosenblattBicop(
        **config,
        transform="logit",
        device=DEVICE,
      )
    )
    model.fit(uv_train, x=x_train)
    models[label] = model
    print(f"{label:20s} fitted in {perf_counter() - started:7.2f} s")
  except (ImportError, RuntimeError) as exc:
    print(f"{label:20s} skipped: {type(exc).__name__}: {exc}")

if not models:
  raise RuntimeError("No backend could be fitted; inspect the messages above.")

## Conditional density across x

Each panel fixes a point $(u,v)$ and sweeps $x$. The dashed black curve is the exact Clayton density under $\tau(x)$; colored curves are estimates from the fitted backends.

In [ ]:
UV_PAIRS = np.array(
  [[0.20, 0.20], [0.20, 0.80], [0.50, 0.50], [0.80, 0.80]],
  dtype=np.float64,
)
X_EVAL = np.linspace(X_MIN, X_MAX, 50)

truth_pdf = np.column_stack(
  [clayton_at_tau(float(tau)).pdf(UV_PAIRS) for tau in tau_of_x(X_EVAL)]
)
uv_eval = np.repeat(UV_PAIRS, X_EVAL.size, axis=0)
x_eval = np.tile(X_EVAL, UV_PAIRS.shape[0])[:, None]
estimated_pdf = {
  label: np.asarray(model.pdf(uv_eval, x_eval)).reshape(
    UV_PAIRS.shape[0], X_EVAL.size
  )
  for label, model in models.items()
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for pair_idx, (ax, (u, v)) in enumerate(zip(axes.flat, UV_PAIRS)):
  ax.plot(X_EVAL, truth_pdf[pair_idx], "k--", lw=2.5, label="truth")
  for label, values in estimated_pdf.items():
    ax.plot(X_EVAL, values[pair_idx], lw=1.8, label=label)
  ax.set_title(f"u={u:.2f}, v={v:.2f}")
  ax.set_ylabel("conditional density")
  ax.grid(alpha=0.25)
for ax in axes[-1]:
  ax.set_xlabel("x")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=4)
fig.suptitle("Conditional Clayton density: backend comparison")
fig.tight_layout(rect=(0, 0.08, 1, 0.96))
plt.show()

## Recovered dependence curve

`tau(x_row=...)` simulates from the fitted conditional copula at a fixed covariate row and computes Kendall's tau. It is therefore more expensive than density evaluation; a modest evaluation grid is enough for this diagnostic.

In [ ]:
X_TAU = np.linspace(0.05, 0.95, 11)
tau_estimates = {
  label: np.array([model.tau(x_row=np.array([[x]]), n=750) for x in X_TAU])
  for label, model in models.items()
}

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(X_TAU, tau_of_x(X_TAU), "k--", lw=2.5, label="truth")
for label, values in tau_estimates.items():
  ax.plot(X_TAU, values, marker="o", label=label)
ax.set(
  xlabel="x",
  ylabel="Kendall's tau",
  title="True and estimated conditional dependence",
  ylim=(0, 1),
)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Density slices on standard-normal margins

The final comparison is shown after transforming both copula margins to standard normals: $z_j=\Phi^{-1}(u_j)$. This changes both the axes and the density. If $c$ is the copula density, the density on normal margins is

$$f_{Z_1,Z_2}(z_1,z_2\mid x)=c\{\Phi(z_1),\Phi(z_2)\mid x\}\,\phi(z_1)\phi(z_2).$$

Rows correspond to selected values of $x$; the first column is the exact Clayton density and subsequent columns are fitted backends.

In [ ]:
GRID_SIZE = 35
normal_axis = np.linspace(norm.ppf(0.02), norm.ppf(0.98), GRID_SIZE)
copula_axis = norm.cdf(normal_axis)
uu, vv = np.meshgrid(copula_axis, copula_axis, indexing="ij")
grid_uv = np.column_stack([uu.ravel(), vv.ravel()])
normal_jacobian = np.outer(norm.pdf(normal_axis), norm.pdf(normal_axis))
X_SLICES = [0.15, 0.50, 0.85]
column_names = ["Truth", *models]

fig, axes = plt.subplots(
  len(X_SLICES),
  len(column_names),
  figsize=(4 * len(column_names), 10),
  sharex=True,
  sharey=True,
  squeeze=False,
)
for row, x in enumerate(X_SLICES):
  tau = float(tau_of_x(np.array([x]))[0])
  surfaces = [
    clayton_at_tau(tau).pdf(grid_uv).reshape(GRID_SIZE, GRID_SIZE)
    * normal_jacobian,
    *[
      np.asarray(
        model.pdf_grid(copula_axis, copula_axis, x_row=np.array([[x]]))
      )
      * normal_jacobian
      for model in models.values()
    ],
  ]
  vmax = max(float(np.quantile(surface, 0.98)) for surface in surfaces)
  for col, (ax, title, surface) in enumerate(
    zip(axes[row], column_names, surfaces)
  ):
    image = ax.contourf(
      normal_axis,
      normal_axis,
      surface.T,
      levels=20,
      vmin=0,
      vmax=vmax,
    )
    if row == 0:
      ax.set_title(title)
    if col == 0:
      ax.set_ylabel(f"x={x:.2f}\n$\\tau$={tau:.2f}\n$z_2$")
    if row == len(X_SLICES) - 1:
      ax.set_xlabel("$z_1$")
    ax.set_aspect("equal")
  fig.colorbar(image, ax=axes[row].tolist(), shrink=0.75, label="density")
fig.suptitle("Conditional density on standard-normal margins", y=1.01)
plt.show()